패키지

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
from scipy.stats import norm, shapiro, kstest, probplot, jarque_bera
import plotly
import plotly.express as px
import nbformat

from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import gc

from collections import Counter

In [2]:
# 모델링 관련 패키지
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
import xgboost as xgb
from xgboost.callback import EarlyStopping

import optuna

from sklearn.metrics import mean_squared_error

c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


한글 패치

In [3]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 나눔고딕 경로 (Windows 설치 위치)
font_path = "C:/Windows/Fonts/NanumGothic.ttf"
fontprop = fm.FontProperties(fname=font_path)

plt.rc('font', family='Malgun Gothic')   # 기본 한글 폰트
plt.rcParams['axes.unicode_minus'] = False

승하차예측 데이터 불러오기 : R만 사용

In [4]:
in_pred = pd.read_csv(r"C:\Users\pc\Desktop\JB\졸업논문\연구기록\1. SARMA + GARCH 예측\SARMA + GARCH - 8시 승차인원 예측.csv", encoding = "cp949")
out_pred = pd.read_csv(r"C:\Users\pc\Desktop\JB\졸업논문\연구기록\1. SARMA + GARCH 예측\SARMA + GARCH - 8시 하차인원 예측.csv", encoding = "cp949")

In [5]:
subway_time = pd.read_csv(r"C:\Users\pc\Desktop\JB\졸업논문\데이터\원본\14. (1~9호선) 열차운행시각표\[260312] 이름변경 - 서울교통공사_서울 도시철도 열차운행시각표_20250704.csv", encoding = "cp949")

C:\Users\pc\AppData\Local\Temp\ipykernel_28332\3065937938.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  subway_time = pd.read_csv(r"C:\Users\pc\Desktop\JB\졸업논문\데이터\원본\14. (1~9호선) 열차운행시각표\[260312] 이름변경 - 서울교통공사_서울 도시철도 열차운행시각표_20250704.csv", encoding = "cp949")


In [7]:
subway_time = subway_time[subway_time['역사명'].isin(in_pred.columns)]

In [8]:
# 시간을 24시, 25시, 이렇게 기록한거 수정
def fix_hour(series):
    s = series.astype(str)

    # 시, 분, 초 분리
    parts = s.str.split(':', expand=True)

    hour = pd.to_numeric(parts[0], errors='coerce')
    minute = parts[1]
    second = parts[2]

    # 24 이상이면 24 빼기
    hour = hour.where(hour < 24, hour - 24)

    # 다시 문자열 합치기
    return hour.astype('Int64').astype(str).str.zfill(2) + ':' + minute + ':' + second

subway_time['열차도착시간'] = fix_hour(subway_time['열차도착시간'])
subway_time['열차출발시간'] = fix_hour(subway_time['열차출발시간'])

In [9]:
import pandas as pd

# 시간 컬럼 datetime 변환
subway_time['열차도착시간'] = pd.to_datetime(subway_time['열차도착시간'], format='%H:%M:%S', errors='coerce')
subway_time['열차출발시간'] = pd.to_datetime(subway_time['열차출발시간'], format='%H:%M:%S', errors='coerce')

# 1. 둘 다 NaN인 행 삭제
# subway_time = subway_time.dropna(subset=['열차도착시간', '열차출발시간'], how='all')

# 2. 도착시간만 NaN → 출발시간 - 30초
mask_arrival_nan = subway_time['열차도착시간'].isna() & subway_time['열차출발시간'].notna()
subway_time.loc[mask_arrival_nan, '열차도착시간'] = (
    subway_time.loc[mask_arrival_nan, '열차출발시간'] - pd.Timedelta(seconds=30)
)

# 3. 출발시간만 NaN → 도착시간 + 30초
mask_departure_nan = subway_time['열차출발시간'].isna() & subway_time['열차도착시간'].notna()
subway_time.loc[mask_departure_nan, '열차출발시간'] = (
    subway_time.loc[mask_departure_nan, '열차도착시간'] + pd.Timedelta(seconds=30)
)

# 다시 HH:MM:SS 형태로 변환 (원하면)
subway_time['열차도착시간'] = subway_time['열차도착시간'].dt.strftime('%H:%M:%S')
subway_time['열차출발시간'] = subway_time['열차출발시간'].dt.strftime('%H:%M:%S')

In [10]:
subway_time

,고유번호,호선,역사코드,역사명,주중주말,방향,급행여부,열차코드,열차도착시간,열차출발시간,출발역,도착역
0,1,1,150,서울역_1호선,DAY,DOWN,0,K101,12:53:00,12:53:30,양주,인천
1,2,1,150,서울역_1호선,DAY,DOWN,0,K103,13:02:30,13:03:00,소요산,인천
2,3,1,150,서울역_1호선,DAY,DOWN,0,K107,13:25:00,13:25:30,의정부,인천
3,4,1,150,서울역_1호선,DAY,DOWN,0,K11,05:30:00,05:30:30,광운대,인천
4,5,1,150,서울역_1호선,DAY,DOWN,0,K111,13:48:00,13:48:30,의정부,인천
...,...,...,...,...,...,...,...,...,...,...,...,...
389516,389517,8,2828,남위례_8호선,SAT,UP,0,8234,22:54:00,22:54:30,모란,별내
389517,389518,8,2828,남위례_8호선,SAT,UP,0,8236,23:06:00,23:06:30,모란,별내
389518,389519,8,2828,남위례_8호선,SAT,UP,0,8238,23:21:00,23:21:30,모란,별내
389519,389520,8,2828,남위례_8호선,SAT,UP,0,8240,23:36:00,23:36:30,모란,암사


In [11]:
len(subway_time['역사명'].unique())

269

In [12]:
# 딕셔너리로 저장
subway_dict = {name: df for name, df in subway_time.groupby('역사명')}
subway_dict['구산_6호선']

,고유번호,호선,역사코드,역사명,주중주말,방향,급행여부,열차코드,열차도착시간,열차출발시간,출발역,도착역
294043,294044,6,2616,구산_6호선,DAY,DOWN,0,6009,05:36:10,05:36:40,독바위,봉화산
294044,294045,6,2616,구산_6호선,DAY,DOWN,0,6011,05:47:00,05:47:30,응암,신내
294045,294046,6,2616,구산_6호선,DAY,DOWN,0,6013,05:57:50,05:58:20,응암,신내
294046,294047,6,2616,구산_6호선,DAY,DOWN,0,6015,06:07:50,06:08:20,응암,신내
294047,294048,6,2616,구산_6호선,DAY,DOWN,0,6017,06:15:30,06:16:00,응암,봉화산
...,...,...,...,...,...,...,...,...,...,...,...,...
294445,294446,6,2616,구산_6호선,SAT,DOWN,0,6241,23:06:40,23:07:10,응암,봉화산
294446,294447,6,2616,구산_6호선,SAT,DOWN,0,6243,23:17:30,23:18:00,응암,안암
294447,294448,6,2616,구산_6호선,SAT,DOWN,0,6245,23:29:00,23:29:30,응암,한강진
294448,294449,6,2616,구산_6호선,SAT,DOWN,0,6247,23:39:30,23:40:00,응암,공덕


In [13]:
hour = subway_dict['구산_6호선'].loc[subway_dict['구산_6호선']['주중주말'] == 'DAY']['열차도착시간'].str.split(':').str[0].astype(int)
(hour == 8).sum()

np.int64(15)

In [14]:
result_8h = {}

for station, df in subway_dict.items():
    hour = df.loc[df['주중주말'] == 'DAY']['열차도착시간'].str.split(':').str[0].astype(int)
    result_8h[station] = (hour == 8).sum()

result_8h = pd.Series(result_8h).sort_index()
result_8h['구산_6호선']

np.int64(15)

In [15]:
# 8시~8시59분 열차수 산출
result_8h

가락시장_3호선           24
가락시장_8호선           23
가산디지털단지_7호선        31
강남_2호선             41
강남구청_7호선           35
                   ..
홍제_3호선             32
화곡_5호선             32
화랑대.서울여대입구._6호선    24
회현.남대문시장._4호선      35
효창공원앞_6호선          29
Length: 269, dtype: int64

In [16]:
# x_s 산출!!!
x = (in_pred.loc[0].sort_index() + out_pred.loc[0].sort_index()) / result_8h
x

가락시장_3호선            83.324012
가락시장_8호선           102.005609
가산디지털단지_7호선        534.513186
강남_2호선             388.677798
강남구청_7호선           136.077153
                      ...    
홍제_3호선             117.831108
화곡_5호선             217.706305
화랑대.서울여대입구._6호선    108.738543
회현.남대문시장._4호선      197.986838
효창공원앞_6호선           75.655196
Length: 269, dtype: float64

In [17]:
value_min = min(x)
value_max = max(x)

In [18]:
value_max

534.5131860376595

In [19]:
x.idxmin()
x.idxmax()

'가산디지털단지_7호선'

In [20]:
xhat = (x - value_min) / (value_max - value_min)
xhat

가락시장_3호선           0.142481
가락시장_8호선           0.177987
가산디지털단지_7호선        1.000000
강남_2호선             0.722829
강남구청_7호선           0.242743
                     ...   
홍제_3호선             0.208065
화곡_5호선             0.397885
화랑대.서울여대입구._6호선    0.190783
회현.남대문시장._4호선      0.360407
효창공원앞_6호선          0.127906
Length: 269, dtype: float64

In [21]:
xhat['잠실.송파구청._2호선']

np.float64(0.6647927434147187)

In [22]:
y = (out_pred.loc[0].sort_index())/(in_pred.loc[0].sort_index() + out_pred.loc[0].sort_index())
y

가락시장_3호선           0.549421
가락시장_8호선           0.749267
가산디지털단지_7호선        0.930563
강남_2호선             0.835498
강남구청_7호선           0.915020
                     ...   
홍제_3호선             0.260807
화곡_5호선             0.188765
화랑대.서울여대입구._6호선    0.236844
회현.남대문시장._4호선      0.951101
효창공원앞_6호선          0.474135
Name: 0, Length: 269, dtype: float64

In [76]:
# 몇개 역만 조정해주기
xhat['성수E_2호선'] = xhat['성수_2호선']
xhat['신도림E_2호선'] = xhat['신도림_2호선']
xhat['강동(마천)_5호선'] = xhat['강동_5호선']
xhat['응암S_6호선'] = xhat['응암_6호선']

y['성수E_2호선'] = y['성수_2호선']
y['신도림E_2호선'] = y['신도림_2호선']
y['강동(마천)_5호선'] = y['강동_5호선']
y['응암S_6호선'] = y['응암_6호선']

# 열차이용피로도지수 산출

In [77]:
# 혼잡도데이터 불러오기
conjest = pd.read_csv(r"C:\Users\pc\Desktop\JB\졸업논문\데이터\원본\1. (1~8호선) 분기별_지하철 시간대별 혼잡도정보\[260313] 8시 역이름변경 - 서울교통공사_지하철혼잡도정보_20250630.csv", encoding = "cp949")

# %로 변환
conjest['8시 혼잡도'] = conjest['8시 혼잡도'] * 0.01

In [87]:
# xhat 값을 출발역 기준으로 매핑
conjest['xhat'] = conjest['출발역'].map(xhat)
conjest['y'] = conjest['출발역'].map(y)

In [91]:
conjest[conjest['xhat'].isna()]

,요일구분,호선,역번호,출발역,상하구분,8시 혼잡도,xhat,y,열차이용피로도지수


In [89]:
# NaN값인 행들 제거
conjest = conjest.dropna(subset=['xhat'])

In [92]:
# 하이퍼파라미터
alpha = 0.5
beta = 0.5
mu = 0.2

In [94]:
# 열차이용피로도지수 산출
conjest['열차이용피로도지수'] = alpha * conjest['8시 혼잡도'] + beta * conjest['xhat'] - mu * conjest['y']

# csv로 내보내기
pd.DataFrame(conjest).to_csv(
    r"C:\Users\pc\Desktop\JB\졸업논문\연구기록\열차이용피로도지수.csv",
    encoding="utf-8-sig"
)

conjest

C:\Users\pc\AppData\Local\Temp\ipykernel_28332\1454839242.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  conjest['열차이용피로도지수'] = alpha * conjest['8시 혼잡도'] + beta * conjest['xhat'] - mu * conjest['y']


,요일구분,호선,역번호,출발역,상하구분,8시 혼잡도,xhat,y,열차이용피로도지수
0,평일,1호선,150,서울역_1호선,상선,0.8030,0.845364,0.644999,0.695182
1,평일,1호선,150,서울역_1호선,하선,0.3160,0.845364,0.644999,0.451682
2,평일,1호선,151,시청_1호선,상선,0.5495,0.486061,0.961477,0.325485
3,평일,1호선,151,시청_1호선,하선,0.3010,0.486061,0.961477,0.201235
4,평일,1호선,152,종각_1호선,상선,0.3090,0.641010,0.969343,0.281137
...,...,...,...,...,...,...,...,...,...
548,평일,8호선,2825,신흥_8호선,하선,0.1715,0.081058,0.247300,0.076819
549,평일,8호선,2826,수진_8호선,상선,0.1165,0.096487,0.373527,0.031788
550,평일,8호선,2826,수진_8호선,하선,0.1715,0.096487,0.373527,0.059288
551,평일,8호선,2827,모란_8호선,상선,0.0785,0.061585,0.527798,-0.035517
